# National Ecosystem Accounting Project (NEAP) IUCN layers

Ecosystem extent layers for Australia, methods developed by @Macfarlane_2024.
 
 
## Data
Macfarlane C, Cox S, Francis RJ, Jordan R, Keith DA, Kingsford R, Liu N, Newnham GJ, Nicholson E, Prober SM, Richards AE, Tetreault-Campbell S, Trebilco R and Schmidt RK (2024) Spatial extent of IUCN ecosystem functional groups at 250 m resolution for freshwater, terrestrial, marine and transitional realms for Australia: pre-1750, 2010-11, 2015-16 and 2020-21. CSIRO. Data Collection. https://data.csiro.au/collection/csiro:63068
 


## Download data from data.csiro.au
### Using python

Instruction to work with CSIRO data in Python are available at: https://bitbucket.csiro.au/projects/RDS/repos/examples/browse

In [1]:
import pyprojroot
import os
from pathlib import Path
import sys
import json
import requests

In [2]:
repodir = pyprojroot.find_root(pyprojroot.has_dir(".git"))
sys.path.append(str(repodir))

In [4]:
from lib.csiro.encodeID import encodeIdentifier

In [5]:
baseURL = "https://data.csiro.au/dap/ws/v2/"
endpoint = "collections/{id}"
collectionURL = "https://doi.org/10.25919/t8fc-er82"
encodedID = encodeIdentifier(collectionURL)
url = baseURL + endpoint.format(id=encodedID)

In [6]:
headers = {"Accept":"application/json"}

r = requests.get(url, headers=headers)
print("Collection metadata URL: {0}\n".format(r.url))

Collection metadata URL: https://data.csiro.au/dap/ws/v2/collections/10.25919/t8fc-er82



In [7]:
resultPage = r.json()   # A dict of the response
# print(json.dumps(resultPage, indent=2))
resultPage.keys()

dict_keys(['dataCollectionTypeCode', 'id', 'dataCollectionId', 'dataCollectionCommonId', 'versionNumber', 'dataVersionNumber', 'self', 'landingPage', 'title', 'description', 'legacyId', 'fieldsOfResearch', 'dataStartDate', 'dataEndDate', 'keywords', 'relatedLinks', 'lineage', 'credit', 'licence', 'licenceLink', 'organisations', 'attributionStatement', 'rights', 'access', 'published', 'leadResearcher', 'versions', 'metadata', 'data', 'serviceCount', 'supportingFiles', 'spatialParameters', 'contributors', 'allNames', 'domainType', 'collectionContentType', 'andsPid', 'doi', 'project', 'organisationalLevels', 'accessLevel', 'dataRestricted', 'depositStatus', 'accessViewable', 'withdrawn', 'blocked', 'askapLevel7', 'permaLink', 'selfLink', 'templateCollection'])

In [8]:
# Get the data endpoint
data_url = resultPage.get("data")
r = requests.get(data_url, headers=headers)
print("Collection data URL: {0}\n".format(r.url))
dataPage = r.json()
#print(json.dumps(dataPage, indent=2))

Collection data URL: https://data.csiro.au/dap/ws/v2/collections/64762/data



In [9]:
# One thing you could do with the file list is create a dict matching
# file names to URLs
fileURLs = {}
files = dataPage.get("file")
for file in files:
    filename = file["filename"]
    fileURL = file["link"]["href"]
    # fileURL = file["presignedLink"]["href"] #Alternative for large collections
    # # Note that presignedLink URLs will expire 48 hours after your requested
    # # the list of files.
    fileURLs[filename] = fileURL

In [10]:
fileURLs.keys()

dict_keys(['Crosswalks/ALUM-IUCNGET.xlsx', 'Crosswalks/Estuarine-IUCNGET.xlsx', 'Crosswalks/Lacustrine-IUCNGET.xlsx', 'Crosswalks/Marine-IUCNGET.xlsx', 'Crosswalks/Pelagic-IUCNGET.xlsx', 'Crosswalks/Riverine-IUCNGET.xlsx', 'Crosswalks/Rivers-IUCNGET.xlsx', 'Crosswalks/Springs-IUCNGET.xlsx', 'Crosswalks/Terrestrial_IUCNGET.xlsx', 'Freshwater Rivers and streams biome/GET_Riverine.cpg', 'Freshwater Rivers and streams biome/GET_Riverine.dbf', 'Freshwater Rivers and streams biome/GET_Riverine.prj', 'Freshwater Rivers and streams biome/GET_Riverine.shp', 'Freshwater Rivers and streams biome/GET_Riverine.shx', 'Geofabric-derived input data/GET_Lakes.cpg', 'Geofabric-derived input data/GET_Lakes.dbf', 'Geofabric-derived input data/GET_Lakes.prj', 'Geofabric-derived input data/GET_Lakes.shp', 'Geofabric-derived input data/GET_Lakes.shx', 'Marine-Freshwater-Terrestrial-Transitional/GET_MarineTerrestrial_2010_250m.tif', 'Marine-Freshwater-Terrestrial-Transitional/GET_MarineTerrestrial_2010_250m.t

In [11]:

# Now if you want a file URL using the file name...
filename = 'Crosswalks/Terrestrial_IUCNGET.xlsx'
fileURL = fileURLs[filename]
print("{0}: {1}".format(filename, fileURL))

Crosswalks/Terrestrial_IUCNGET.xlsx: https://data.csiro.au/dap/ws/v2/collections/64762/data/80512256


In [12]:
filename

'Crosswalks/Terrestrial_IUCNGET.xlsx'

In [13]:
output_dir = repodir / 'gisdata' / 'AUS' / 'NEAP'

In [14]:
output_file = output_dir / filename

In [15]:
output_file.exists()

False

In [16]:
def download_file(url, output_dir, filename):
    output_file = output_dir / filename
    if output_file.exists():
        print('No need to download, file already there')
        return True
    else:
        Path(output_file).parent.mkdir(parents=True, exist_ok=True)
        try:
            # Send a GET request to the URL
            response = requests.get(url, stream=True)
            response.raise_for_status()  # Raises an HTTPError for bad responses
            
            # Open the local file to write the downloaded content
            with open(output_file, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
            return True
        except requests.exceptions.RequestException as e:
            print(f"Error downloading file: {e}")
            return False

In [17]:
success = download_file(fileURL, output_dir, filename)
if success:
    print(f"Download of {filename} completed successfully")

Download of Crosswalks/Terrestrial_IUCNGET.xlsx completed successfully


In [18]:
for fn in fileURLs.keys():
    success = download_file(fileURLs[fn], output_dir, fn)
    if success:
        print(f"Download of {fn} completed successfully")

Download of Crosswalks/ALUM-IUCNGET.xlsx completed successfully
Download of Crosswalks/Estuarine-IUCNGET.xlsx completed successfully
Download of Crosswalks/Lacustrine-IUCNGET.xlsx completed successfully
Download of Crosswalks/Marine-IUCNGET.xlsx completed successfully
Download of Crosswalks/Pelagic-IUCNGET.xlsx completed successfully
Download of Crosswalks/Riverine-IUCNGET.xlsx completed successfully
Download of Crosswalks/Rivers-IUCNGET.xlsx completed successfully
Download of Crosswalks/Springs-IUCNGET.xlsx completed successfully
No need to download, file already there
Download of Crosswalks/Terrestrial_IUCNGET.xlsx completed successfully
Download of Freshwater Rivers and streams biome/GET_Riverine.cpg completed successfully
Download of Freshwater Rivers and streams biome/GET_Riverine.dbf completed successfully
Download of Freshwater Rivers and streams biome/GET_Riverine.prj completed successfully
Download of Freshwater Rivers and streams biome/GET_Riverine.shp completed successfully
